# Retry Metrics Dashboard

This notebook computes key retry metrics from audit logs and supports windowed analysis.

**Metrics:**
- Retry rate: Count of `retry_attempt_failed` events
- Success after retry: Count of `retry_success` events
- Exhausted retries: Count of `retry_exhausted` events
- Hard failures: Count of `hard_failure` events
- Retry by operation: Group counts by `operation`

**Log Format:**
- Supports both new format (top-level fields) and legacy format (nested in metadata)
- Works with both regular and academic ingestion retry logs

In [ ]:
# Imports
from pathlib import Path
import json
from datetime import datetime, timezone, timedelta
import pandas as pd
import sys

## Configuration
Adjust the parameters below to set the audit log location and the analysis window.

In [ ]:
# Parameters
# Path to the audit JSONL file produced by retry logic
# Change if needed
workspace_root = Path(__file__).resolve().parents[1] if '__file__' in globals() else Path.cwd().parents[0]
LOG_PATH = workspace_root / 'logs' / 'retry_audit.jsonl' 

# Window configuration: choose either explicit start/end or a rolling window
USE_ROLLING_WINDOW = True
# ROLLING_MINUTES = 60  # last N minutes
ROLLING_MINUTES = 1440  # last N minutes

# Explicit window (ISO 8601). Example: '2026-01-11T10:00:00Z'
WINDOW_START_ISO = None
WINDOW_END_ISO = None

# If your timestamps end with 'Z', we'll treat them as UTC.
def _parse_iso(ts: str) -> datetime:
    ts = ts.strip()
    if ts.endswith('Z'):
        ts = ts.replace('Z', '+00:00')
    try:
        return datetime.fromisoformat(ts)
    except ValueError:
        # Fallback for common formats without timezone
        return datetime.strptime(ts, '%Y-%m-%dT%H:%M:%S')

def _now_utc() -> datetime:
    return datetime.now(timezone.utc)

def get_window() -> tuple[datetime, datetime]:
    if USE_ROLLING_WINDOW:
        end = _now_utc()
        start = end - timedelta(minutes=ROLLING_MINUTES)
        return start, end
    # Explicit window
    if WINDOW_START_ISO is None:
        raise ValueError('WINDOW_START_ISO must be set when USE_ROLLING_WINDOW=False')
    start = _parse_iso(WINDOW_START_ISO)
    end = _parse_iso(WINDOW_END_ISO) if WINDOW_END_ISO else _now_utc()
    # Normalise to UTC if naive
    if start.tzinfo is None:
        start = start.replace(tzinfo=timezone.utc)
    if end.tzinfo is None:
        end = end.replace(tzinfo=timezone.utc)
    return start, end

## Load and Prepare Data
Reads JSONL audit entries and flattens relevant metadata fields.

In [ ]:
# Load audit file
def load_audit_df(path: Path) -> pd.DataFrame:
    if not path.exists():
        print(f'Audit file not found: {path}')
        return pd.DataFrame(columns=['event','timestamp','operation','attempt','max_retries','error_type','error_message','failure_type','will_retry'])
    rows = []
    with path.open('r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                obj = json.loads(line)
            except json.JSONDecodeError:
                # Skip malformed line
                continue
            event = obj.get('event')
            ts = obj.get('timestamp')
            md = obj.get('metadata', {}) or {}
            
            # Support both new format (top-level fields) and old format (nested in metadata)
            # Try top-level first, fall back to metadata
            rows.append({
                'event': event,
                'timestamp': ts,
                'operation': obj.get('operation') or md.get('operation'),
                'attempt': obj.get('attempt') or md.get('attempt'),
                'max_retries': obj.get('max_retries') or md.get('max_retries'),
                'error_type': obj.get('error_type') or md.get('error_type'),
                'error_message': obj.get('error_message') or md.get('error_message'),
                'failure_type': obj.get('failure_type') or md.get('failure_type'),
                'will_retry': obj.get('will_retry') if obj.get('will_retry') is not None else md.get('will_retry'),
            })
    df = pd.DataFrame(rows)
    if not df.empty:
        # Parse timestamps to timezone-aware datetimes
        df['timestamp_dt'] = df['timestamp'].apply(lambda x: _parse_iso(x) if isinstance(x, str) else None)
        df['timestamp_dt'] = df['timestamp_dt'].apply(lambda d: d if d and d.tzinfo else (d.replace(tzinfo=timezone.utc) if d else None))
        # Fill missing
        df['operation'] = df['operation'].fillna('unknown')
    return df

audit_df = load_audit_df(LOG_PATH)
print(f'Loaded {len(audit_df)} audit entries from {LOG_PATH}')
audit_df.head(3)

## Filter by Window
Compute the date window and filter the DataFrame accordingly.

In [ ]:
start_dt, end_dt = get_window()
print('Window start:', start_dt.isoformat())
print('Window end  :', end_dt.isoformat())

if audit_df.empty:
    window_df = audit_df.copy()
else:
    window_df = audit_df[(audit_df['timestamp_dt'] >= start_dt) & (audit_df['timestamp_dt'] <= end_dt)].copy()
print(f'Windowed entries: {len(window_df)}')
window_df.head(3)

## Metrics
Counts for each key metric and grouping by operation.

In [ ]:
def count_event(df: pd.DataFrame, name: str) -> int:
    if df.empty:
        return 0
    return int((df['event'] == name).sum())

metrics = {
    'retry_rate': count_event(window_df, 'retry_attempt_failed'),
    'success_after_retry': count_event(window_df, 'retry_success'),
    'exhausted_retries': count_event(window_df, 'retry_exhausted'),
    'hard_failures': count_event(window_df, 'hard_failure'),
}
metrics_df = pd.DataFrame([metrics])
metrics_df.T.rename(columns={0: 'count'})

In [ ]:
# Group by operation for retry-related events
if window_df.empty:
    by_op_attempts = pd.DataFrame(columns=['operation','count'])
    by_op_success = pd.DataFrame(columns=['operation','count'])
    by_op_exhausted = pd.DataFrame(columns=['operation','count'])
else:
    attempts = window_df[window_df['event'] == 'retry_attempt_failed']
    success = window_df[window_df['event'] == 'retry_success']
    exhausted = window_df[window_df['event'] == 'retry_exhausted']
    by_op_attempts = attempts.groupby('operation').size().reset_index(name='count').sort_values('count', ascending=False)
    by_op_success = success.groupby('operation').size().reset_index(name='count').sort_values('count', ascending=False)
    by_op_exhausted = exhausted.groupby('operation').size().reset_index(name='count').sort_values('count', ascending=False)

print('Attempts by operation:')
print(by_op_attempts.to_string(index=False) if not by_op_attempts.empty else '  (none)')
print()

print('Success by operation:')
print(by_op_success.to_string(index=False) if not by_op_success.empty else '  (none)')
print()

print('Exhausted by operation:')
print(by_op_exhausted.to_string(index=False) if not by_op_exhausted.empty else '  (none)')

### Notes
- Log file defaults to `logs/retry_audit.jsonl` which contains retry events from all ingestion types (regular, academic, code)
- Adjust `LOG_PATH` if your audit file is located elsewhere
- Use `USE_ROLLING_WINDOW=False` and set `WINDOW_START_ISO`/`WINDOW_END_ISO` for explicit ranges
- Timestamp parsing supports trailing 'Z' and ISO offsets
- Supports both new log format (top-level fields) and legacy format (nested in metadata)